# 🎨 AI娘化双角色同框实验：猫吃鱼 & 鱼骑龙 (Qwen-Image-2.1)

使用阿里开源旗舰 **Qwen-Image-2.1 (`Qwen/Qwen-Image-2.1`)** 进行多图角色一致性融合：
1. **猫吃鱼**：Gemini 猫娘 × DeepSeek 鲸鱼娘
2. **鱼骑龙**：DeepSeek 鲸鱼娘 × ChatGPT 龙娘

> 推荐运行环境：Colab **A100** 或 **L4** GPU 运行时。

In [ ]:
#@title 1. 挂载 Google Drive 并检查 GPU
import os
from google.colab import drive

drive.mount('/content/drive')

WORKSPACE = '/content/drive/MyDrive/AIGC_Workspace'
DATASETS_DIR = os.path.join(WORKSPACE, 'datasets')
OUTPUTS_DIR = os.path.join(WORKSPACE, 'outputs')
MODELS_DIR = os.path.join(WORKSPACE, 'models', 'image')

os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

!nvidia-smi

In [ ]:
#@title 2. 安装最新 Diffusers 与依赖
!pip install -q --upgrade torch torchvision transformers accelerate pillow
!pip install -q git+https://github.com/huggingface/diffusers

In [ ]:
#@title 3. 准备角色参考图（优先读取 Drive，缺失则弹窗上传）
import glob
from PIL import Image
from google.colab import files

def find_or_upload(keyword, fallback_name):
    matches = glob.glob(os.path.join(DATASETS_DIR, f'*{keyword}*'))
    if matches:
        print(f'✅ 找到云盘素材: {matches[0]}')
        return matches[0]
    print(f'⚠️ 云盘 {DATASETS_DIR} 中未找到包含 "{keyword}" 的图片，请手动上传 {fallback_name}...')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    dst = os.path.join(DATASETS_DIR, fname)
    os.replace(fname, dst)
    print(f'✅ 已保存至云盘: {dst}')
    return dst

whale_path = find_or_upload('DeepSeek', '01_DeepSeek_深度求索(鲸鱼娘).png')
cat_path = find_or_upload('Gemini', '05_Gemini_谷歌双子座(猫娘).jpg')
dragon_path = find_or_upload('ChatGPT', '07_ChatGPT_OpenAI(龙娘与Sora).jpg')

img_whale = Image.open(whale_path).convert('RGB')
img_cat = Image.open(cat_path).convert('RGB')
img_dragon = Image.open(dragon_path).convert('RGB')
print('🎉 三张角色立绘加载完成！')

In [ ]:
#@title 4. 加载 Qwen-Image-2.1 模型（自动缓存到 Drive）
import torch
from diffusers import QwenImage21Pipeline

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
pipe = QwenImage21Pipeline.from_pretrained(
    'Qwen/Qwen-Image-2.1',
    torch_dtype=dtype,
    cache_dir=MODELS_DIR
)
pipe.enable_model_cpu_offload()
print('🚀 Qwen-Image-2.1 加载完毕！')

In [ ]:
#@title 5. 生成第 1 张：【猫吃鱼】Gemini 猫娘 × DeepSeek 鲸鱼娘
from IPython.display import display

prompt_cat_fish = (
    "A lively anime illustration featuring both characters from the two reference images together in one scene. "
    "On the left, the playful cyber cat-girl with heterochromia eyes (pink and golden) and blue-purple four-pointed star hairpins "
    "is pouncing forward with an excited grin, playfully chomping at the huge blue whale tail of the blue-haired maid girl on the right. "
    "The blue-haired whale maid girl wears her blue-and-white maid outfit with the small blue whale apron pattern, looking surprised and flustered. "
    "Cohesive vibrant 2D anime art style, clean lineart, dynamic comedic interaction, bright studio lighting."
)

out_cat_fish = pipe(
    prompt=prompt_cat_fish,
    image=[img_cat, img_whale],
    width=1280,
    height=720,
    num_inference_steps=30,
    generator=torch.Generator(device='cuda').manual_seed(42)
).images[0]

save_path_1 = os.path.join(OUTPUTS_DIR, 'cat_eats_fish_qwen21.png')
out_cat_fish.save(save_path_1)
print(f'✅ 已保存: {save_path_1}')
display(out_cat_fish)

In [ ]:
#@title 6. 生成第 2 张：【鱼骑龙】DeepSeek 鲸鱼娘 × ChatGPT 龙娘
prompt_fish_dragon = (
    "An epic vibrant anime illustration combining both characters from the two reference images in the sky above fluffy clouds. "
    "The blue-haired whale maid girl with her large blue whale tail and maid apron is cheerfully riding on the back of the regal pure-white dragon girl. "
    "The white dragon girl has white dragon horns, large white dragon wings spread wide in flight, a scaly white dragon tail, and wears her sleek white slit dress with black trim. "
    "A cute little blue cloud with eyes floats beside them. "
    "Cohesive 2D anime illustration, crisp details, bright sky, dynamic sense of flying."
)

out_fish_dragon = pipe(
    prompt=prompt_fish_dragon,
    image=[img_whale, img_dragon],
    width=1280,
    height=720,
    num_inference_steps=30,
    generator=torch.Generator(device='cuda').manual_seed(1024)
).images[0]

save_path_2 = os.path.join(OUTPUTS_DIR, 'fish_rides_dragon_qwen21.png')
out_fish_dragon.save(save_path_2)
print(f'✅ 已保存: {save_path_2}')
display(out_fish_dragon)